In [1]:
# SEU Package Example
# Using the simplified SEU package for entity alignment

import numpy as np
import tensorflow as tf
import seu  # Import the new SEU package
import os

seed = 12345
np.random.seed(seed)

# Choose the GPU, "-1" represents using the CPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

print("SEU Entity Alignment Example using the new package")

SEU Entity Alignment Example using the new package


## Step 1: Load Word Embeddings

Using the SEU package's simplified embedding loading functionality.

In [2]:
# Download and load GloVe embeddings using the SEU package
print("Loading word embeddings...")

# Try to load existing embeddings first
import os
if os.path.exists("glove.6B.300d.txt"):
    print("Loading existing embeddings...")
    word_vecs = seu.load_word_vectors("glove.6B.300d.txt")
else:
    print("Downloading GloVe embeddings...")
    # Download embeddings using the SEU package
    embedding_path = seu.download_and_extract_file(
        "http://nlp.stanford.edu/data/glove.6B.zip",
        "glove.6B.300d.txt",
        dest_path="./glove.6B.300d.txt"
    )
    word_vecs = seu.load_word_vectors(embedding_path)

print(f"Loaded {len(word_vecs)} word vectors")

Loading word embeddings...
Loading existing embeddings...


100%|██████████| 400000/400000 [00:13<00:00, 30283.27it/s]

Loaded 400000 word vectors


## Step 2: Load Dataset

Loading knowledge graph data and entity names using SEU package functions.

In [3]:
# Load knowledge graph data using SEU package
print("Loading dataset...")

# Load triples with reverse edges
file_path = "KGs/dbp_ja_en/"
all_triples, node_size, rel_size = seu.load_triples(file_path, reverse=True)

# Load aligned pairs (use ratio=0 to use all pairs for testing)
train_pair, test_pair = seu.load_aligned_pair(file_path, ratio=0)

# Load entity names
ent_names = seu.load_entity_names("translated_ent_name/dbp_ja_en.json")

print(f"Loaded {len(all_triples)} triples, {node_size} nodes, {rel_size} relations")
print(f"Test pairs: {len(test_pair)}, Entity names: {len(ent_names)}")

Loading dataset...
Loaded 341396 triples, 39594 nodes, 4904 relations
Test pairs: 15000, Entity names: 39594


## Step 3: Generate Features

Using SEU package to generate hybrid features (word-level + character-level).

In [4]:
# Generate features using the SEU package
print("Generating features...")

# Generate hybrid features (combines word-level and character-level features)
feature = seu.generate_features(
    entity_names=ent_names,
    word_vectors=word_vecs,
    node_size=node_size,
    mode="hybrid-level"  # Options: "word-level", "char-level", "hybrid-level"
)

print(f"Generated features with shape: {feature.shape}")

Generating features...
Generated features with shape: (39594, 1841)


## Step 4: Build Graph and Calculate Similarities

Building the sparse adjacency matrix and calculating similarities with feature propagation.

In [5]:
%%time
# Build sparse adjacency matrix using SEU package
print("Building graph structure...")
sparse_rel_matrix = seu.build_sparse_adjacency_matrix(all_triples, node_size)

# Calculate similarities with feature propagation (depth=2 as in original)
print("Calculating similarities with feature propagation...")
sims = seu.calculate_similarities(
    test_pair=test_pair,
    feature=feature, 
    sparse_rel_matrix=sparse_rel_matrix,
    depth=2
)

print(f"Calculated similarity matrix with shape: {sims.shape}")

Building graph structure...
Calculating similarities with feature propagation...
Calculated similarity matrix with shape: (15000, 15000)
CPU times: total: 3min 32s
Wall time: 8.88 s


## Step 5: Entity Alignment Solvers

Running both Hungarian algorithm and Sinkhorn operations using the SEU package.

In [6]:
%%time
# Hungarian Algorithm - exact optimal assignment
print("Running Hungarian algorithm...")
hungarian_result = seu.hungarian_solve(sims)
print("Hungarian algorithm results:")
seu.test(hungarian_result, "hungarian")

Running Hungarian algorithm...
Hungarian algorithm results:
hits@1 : 96.34%
CPU times: total: 11.7 s
Wall time: 11.8 s


In [7]:
%%time
# Sinkhorn Operations - iterative soft assignment
print("Running Sinkhorn algorithm...")
sinkhorn_result = seu.sinkhorn_solve(
    sims, 
    temperature=50.0,    # Temperature scaling parameter
    max_iterations=10    # Number of Sinkhorn iterations
)
print("Sinkhorn algorithm results:")
seu.test(sinkhorn_result, "sinkhorn")

Running Sinkhorn algorithm...
Sinkhorn algorithm results:
hits@1 : 95.83% hits@10 : 98.86% MRR : 96.98%
CPU times: total: 1min 24s
Wall time: 4.13 s
